# 🎓 Student Engagement Detection — Lightweight Temporal Models Experiment

This experiment replaces the heavy **Full Self-Attention Transformer Encoder** (~300k+ parameters) with 3 lightweight temporal architectures optimized for small datasets (~100 video clips):

1. **Attention Pooling** (`--model attention_pool`) — Single learned query vector attending over frame features (~30k-40k params).
2. **1-2 Layer TCN** (`--model tcn`) — Temporal Convolutional Network with 1D Convolutions (~30k params).
3. **Single-Layer GRU** (`--model gru`) — Lightweight unidirectional GRU (~35k params).

---

### Step 1: Mount Google Drive & Set Working Directory

In [ ]:
from google.colab import drive
import os
import sys

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define Project Directories
PROJECT_ROOT = "/content/drive/MyDrive/Student_Engagement_DL"
EXPERIMENT_DIR = os.path.join(PROJECT_ROOT, "colab_lightweight_temporal")

# 3. Navigate into lightweight temporal experiment folder
os.chdir(EXPERIMENT_DIR)
sys.path.append(EXPERIMENT_DIR)
print("Current Working Directory:", os.getcwd())

### Step 2: Install Dependencies

In [ ]:
!pip install -q torch torchvision pandas scikit-learn matplotlib seaborn opencv-python-headless tqdm

### Step 3: Feature Extraction Pipeline (Run Separately from Scratch)

Run the cells below to extract features step-by-step from raw dataset videos.

#### 3a: Extract Video Frames (`Dataset/ -> preprocessed_frames/`)

In [ ]:
!python preprocess/extract_frames.py

#### 3b: Extract EfficientNet-B0 Spatial Features (`preprocessed_frames/ -> efficientnet_features/`)

In [ ]:
!python preprocess/extract_efficientnet.py

#### 3c: Extract OpenFace Facial Features (`preprocessed_frames/ -> openface_features/`)

In [ ]:
!python preprocess/extract_openface.py

#### 3d: Fuse OpenFace + EfficientNet Features into Sequence Tensors (`sequence_fused_features_lightweight/`)

In [ ]:
!python preprocess/fuse_sequence_features.py

### Step 4: Run Pipeline Verification Test

In [ ]:
# Verify forward & backward passes for Attention Pooling, TCN, and GRU architectures
!python test_pipeline_lightweight.py

### Step 5: Train Model Variants
Train any of the 3 lightweight model options below:

#### 🔹 Option A: Attention Pooling (Recommended First)

In [ ]:
!python train_lightweight.py --model attention_pool --epochs 50 --lr 0.001 --batch_size 16

#### 🔹 Option B: 1-2 Layer TCN (Temporal ConvNet)

In [ ]:
!python train_lightweight.py --model tcn --epochs 50 --lr 0.001 --batch_size 16

#### 🔹 Option C: Single-Layer GRU

In [ ]:
!python train_lightweight.py --model gru --epochs 50 --lr 0.001 --batch_size 16

### Step 6: Evaluate Models & Display Metric Reports

In [ ]:
# Evaluate Attention Pooling
!python evaluate_lightweight.py --model attention_pool

In [ ]:
# Evaluate TCN Model
!python evaluate_lightweight.py --model tcn

In [ ]:
# Evaluate Single-Layer GRU
!python evaluate_lightweight.py --model gru

### Step 7: Display Confusion Matrix & Evaluation Results directly in Colab

In [ ]:
from IPython.display import Image, display
import json

for m_name in ["attention_pool", "tcn", "gru"]:
    cm_path = f"evaluation_results_lightweight/confusion_matrix_{m_name}.png"
    metrics_path = f"evaluation_results_lightweight/metrics_{m_name}.json"
    
    if os.path.exists(metrics_path):
        print(f"\n==================== {m_name.upper()} RESULTS ====================")
        with open(metrics_path, "r") as f:
            data = json.load(f)
        print(f"Accuracy:  {data.get('accuracy', 0)*100:.2f}%")
        print(f"Precision: {data.get('precision_weighted', 0)*100:.2f}%")
        print(f"Recall:    {data.get('recall_weighted', 0)*100:.2f}%")
        print(f"F1-Score:  {data.get('f1_weighted', 0)*100:.2f}%")
        
    if os.path.exists(cm_path):
        display(Image(cm_path))